# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is a Metadata object, directly access properties
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Let's discover record sets and their fields, using their `@id` attributes
record_sets = dataset.metadata.record_sets
print("Record sets available (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', '-')}")

# Sample code to list fields for each record set
record_set_fields = {}
for rs in record_sets:
    print(f"\nFields for RecordSet @id = {rs['@id']}:")
    field_list = []
    for field in rs.get('fields', []):
        print(f"  - {field['@id']}: {field.get('name', '-')}")
        field_list.append(field['@id'])
    record_set_fields[rs['@id']] = field_list

# If there are record sets, print a sample record for the first one
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"\nSample record (@id = {example_record_set_id}):")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        if i >= 1:
            break
        print(rec)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Dynamically collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Choose the first record set as main table (update index below as needed)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Main Record Set ID: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis
# Replace with actual @id or column name as found in the overview above

# The typical fields in mass spectrometry tables include: 'Coverage (%)', 'Unique Peptides', 'MW [kDa]', etc.
# Let's try to find a numeric column automatically for this demo:

main_df = dataframes[main_rs_id]
numeric_candidates = main_df.select_dtypes(include=[float, int]).columns.tolist()
if len(numeric_candidates) == 0:
    # Try to convert some likely candidates
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='ignore')
        except Exception:
            pass
    numeric_candidates = main_df.select_dtypes(include=[float, int]).columns.tolist()

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    raise Exception("No numeric fields found for EDA.")

threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalized column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a grouping field
group_field_candidates = [col for col in main_df.columns if col not in numeric_candidates][:1]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field and group_field in main_df:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field and group_field in main_df:
    plt.figure(figsize=(10, 5))
    order = main_df[group_field].value_counts().index
    sns.boxplot(x=group_field, y=numeric_field_id, data=main_df, order=order)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.